### 6.2.2 非线性规划的 Python 求解  
求解非线性规划可以使用 `scipy.optimize` 模块、`cvxopt` 库和 `cvxpy` 库。下面通过一些例子来说明 Python 求解非线性规划的方法。  


### 1. 用 `scipy.optimize` 模块的 `minimize` 函数求解  
**例 6.4** 求解非线性规划问题  
$$
\begin{align*}
\min &\quad \frac{2 + x_1}{1 + x_2} - 3x_1 + 4x_3, \\
\text{s.t.} &\quad 0.1 \leqslant x_i \leqslant 0.9,\quad i = 1,2,3.
\end{align*}
$$  


**解** 利用 Python 软件求得最优解为 $ x_1 = x_2 = 0.9, x_3 = 0.1 $；目标函数的最优值为 $ -0.7737 $。

In [ ]:
from scipy.optimize import minimize
from numpy import ones

def obj(x):
    x1, x2, x3 = x
    return (2 + x1) / (1 + x2) - 3 * x1 + 4 * x3

LB = [0.1] * 3
UB = [0.9] * 3
bound = list(zip(LB, UB))  # 生成决策向量界限的列表
res = minimize(obj, ones(3), bounds=bound)  # 第2个参数为初值

print(res.fun)  # 最优值
print(res.success)  # 求解状态
print(res.x)  # 最优解

-0.7736842105263159
True
[0.9 0.9 0.1]


### 例 6.5  
求解下列非线性规划问题  
$$
\begin{align*}
\max &\quad z = x_1^2 + x_2^2 + 3x_3^2 + 4x_4^2 + 2x_5^2 - 8x_1 - 2x_2 - 3x_3 - x_4 - 2x_5, \\
\text{s.t.} &\quad 
\begin{cases} 
x_1 + x_2 + x_3 + x_4 + x_5 \leqslant 400, \\
x_1 + 2x_2 + 2x_3 + x_4 + 6x_5 \leqslant 800, \\
2x_1 + x_2 + 6x_3 \leqslant 200, \\
x_3 + x_4 + 5x_5 \leqslant 200, \\
0 \leqslant x_i \leqslant 99,\quad i = 1,2,\cdots,5.
\end{cases}
\end{align*}
$$  


**解** 利用 Python 软件，求得最优解  
$$
x_1 = 50.5,\ x_2 = 99,\ x_3 = 0,\ x_4 = 99,\ x_5 = 20.2,
$$  

目标函数的最优值为 $ 51629.93 $。

In [5]:
# 该题只能求得局部最优解，求解时多去几个初值试试
import numpy as np
from scipy.optimize import minimize

c1 = np.array([1, 1, 3, 4, 2])
c2 = np.array([-8, -2, -3, -1, -2])
A = np.array([
    [1, 1, 1, 1, 1],
    [1, 2, 2, 1, 6],
    [2, 1, 6, 0, 0],
    [0, 0, 1, 1, 5]
])
b = np.array([400, 800, 200, 200])
obj = lambda x: -c1 @ x**2 + -c2 @ x  # 求最大值变求最小值
cons = {'type': 'ineq', 'fun': lambda x: b - A @ x}  # type为不是等式，不等式条件用函数表示
bd = [(0, 99) for i in range(A.shape[1])]  # x1~x5都是[0, 99]
res = minimize(obj, np.ones(5) * 90, constraints=cons, bounds=bd)

print(res)  # 输出解的信息

 message: Optimization terminated successfully
 success: True
  status: 0
     fun: -51629.92995523618
       x: [ 5.050e+01  9.900e+01  1.982e-10  9.900e+01  2.020e+01]
     nit: 4
     jac: [-9.300e+01 -1.960e+02  3.000e+00 -7.910e+02 -7.880e+01]
    nfev: 20
    njev: 3


### 2. 用 cvxopt.solvers 模块求解  
第 5 章已经介绍利用 `cvxopt.solvers` 模块求解线性规划模型。这里介绍利用 `cvxopt.solvers` 模块求解二次规划模型。  


**定义 6.3** 若非线性规划的目标函数为决策向量 $ \boldsymbol{x} $ 的二次函数，约束条件又全是线性的，就称这种规划为二次规划。  


`cvxopt.solvers` 模块中二次规划的标准形为  
$$
\begin{align*}
\min &\quad \frac{1}{2} \boldsymbol{x}^{\mathrm{T}} P \boldsymbol{x} + \boldsymbol{q}^{\mathrm{T}} \boldsymbol{x}, \\
\text{s.t.} &\quad 
\begin{cases} 
A \boldsymbol{x} \leqslant \boldsymbol{b}, \\
\text{Aeq} \cdot \boldsymbol{x} = \text{beq}.
\end{cases}
\end{align*}
$$  


**例 6.6** 求解二次规划模型  
$$
\begin{align*}
\min &\quad z = 1.5x_1^2 + x_2^2 + 0.85x_3^2 + 3x_1 - 8.2x_2 - 1.95x_3, \\
\text{s.t.} &\quad 
\begin{cases} 
x_1 + x_3 \leqslant 2, \\
-x_1 + 2x_2 \leqslant 2, \\
x_2 + 2x_3 \leqslant 3, \\
x_1 + x_2 + x_3 = 3.
\end{cases}
\end{align*}
$$  


**解** 先化成标准形，其中  
$$
P = \begin{bmatrix} 
3 & 0 & 0 \\
0 & 2 & 0 \\
0 & 0 & 1.7 
\end{bmatrix}, \quad 
\boldsymbol{q} = \begin{bmatrix} 
3 \\
-8.2 \\
-1.95 
\end{bmatrix}, \quad 
A = \begin{bmatrix} 
1 & 0 & 1 \\
-1 & 2 & 0 \\
0 & 1 & 2 
\end{bmatrix}, \quad 
\boldsymbol{b} = \begin{bmatrix} 
2 \\
2 \\
3 
\end{bmatrix},
$$  

$$
\text{Aeq} = [1, 1, 1], \quad \text{beq} = [3].
$$  


利用 `cvxopt.solvers` 模块，求得最优解为 $ x_1 = 0.8, x_2 = 1.4, x_3 = 0.8 $，目标函数的最优值为 $ -7.1760 $。

In [10]:
from cvxopt import matrix, solvers

n = 3
P = matrix(0.0, (n, n))
P[::n+1] = [3, 2, 1.7]
q = matrix([3.0, -8.2, -1.95])
A = matrix([[1.0, 0.0, 1.0], [-1.0, 2.0, 0.0], [0.0, 1.0, 2.0]]).T  # 注意转置
b = matrix([2.0, 2.0, 3.0])
Aeq = matrix(1.0, (1, n))
beq = matrix([3.0])
s = solvers.qp(P, q, A, b, Aeq, beq)

print(f"最优解为:\n{s['x']}")
print(f"最优值为: {s['primal objective']}")

     pcost       dcost       gap    pres   dres
 0: -1.3148e+01 -4.4315e+00  1e+01  1e+00  9e-01
 1: -6.4674e+00 -7.5675e+00  1e+00  3e-16  4e-16
 2: -7.1538e+00 -7.1854e+00  3e-02  1e-16  2e-15
 3: -7.1758e+00 -7.1761e+00  3e-04  2e-16  2e-15
 4: -7.1760e+00 -7.1760e+00  3e-06  1e-16  2e-15
Optimal solution found.
最优解为:
[ 8.00e-01]
[ 1.40e+00]
[ 8.00e-01]

最优值为: -7.1759977687772745


### 3. 用 cvxpy 库求解  
**例 6.7** 求解下列非线性整数规划问题  
$$
\begin{align*}
\min &\quad z = x_1^2 + x_2^2 + 3x_3^2 + 4x_4^2 + 2x_5^2 - 8x_1 - 2x_2 - 3x_3 - x_4 - 2x_5, \\
\text{s.t.} &\quad 
\begin{cases} 
0 \leqslant x_i \leqslant 99, \text{ 且 } x_i \text{ 为整数}\ (i = 1,\cdots,5), \\
x_1 + x_2 + x_3 + x_4 + x_5 \leqslant 400, \\
x_1 + 2x_2 + 2x_3 + x_4 + 6x_5 \leqslant 800, \\
2x_1 + x_2 + 6x_3 \leqslant 200, \\
x_3 + x_4 + 5x_5 \leqslant 200.
\end{cases}
\end{align*}
$$  


**解** 利用 cvxpy 库，求得的最优解为  
$$
x_1 = 4,\ x_2 = x_5 = 1,\ x_3 = x_4 = 0,
$$  

目标函数的最优值为 $ -17 $。

In [ ]:
import numpy as np
import cvxpy as cp

c1 = np.array([1, 1, 3, 4, 2])
c2 = np.array([-8, -2, -3, -1, -2])
a = np.array([
    [1, 1, 1, 1, 1],
    [1, 2, 2, 1, 6],
    [2, 1, 6, 0, 0],
    [0, 0, 1, 1, 5]
])
b = np.array([400, 800, 200, 200])
x = cp.Variable(5, integer=True)
obj = cp.Minimize(c1 @ x**2 + c2 @ x)
con = [0 <= x, x <= 99, a @ x <= b]
prob = cp.Problem(obj, con)
prob.solve(solver=cp.SCS)

print(f"最优值为: {prob.value}")
print(f"最优解为:\n{x.value}")

SolverError: Problem is mixed-integer, but candidate QP/Conic solvers ([]) are not MIP-capable.

In [40]:
import numpy as np
import cvxpy as cp
from itertools import product

# 定义系数
c1 = np.array([1, 1, 3, 4, 2])
c2 = np.array([-8, -2, -3, -1, -2])

# 约束矩阵和向量
A = np.array([
    [1, 1, 1, 1, 1],
    [1, 2, 2, 1, 6],
    [2, 1, 6, 0, 0],
    [0, 0, 1, 1, 5]
])
b = np.array([400, 800, 200, 200])

print("第一步：求解连续松弛问题...")

# 求解连续问题
x_cont = cp.Variable(5)

# 目标函数
objective_terms = []
for i in range(5):
    objective_terms.append(c1[i] * cp.square(x_cont[i]) + c2[i] * x_cont[i])

objective = cp.Minimize(cp.sum(objective_terms))

# 约束条件
constraints = [
    x_cont >= 0,
    x_cont <= 99,
    A[0] @ x_cont <= b[0],
    A[1] @ x_cont <= b[1],
    A[2] @ x_cont <= b[2],
    A[3] @ x_cont <= b[3]
]

# 使用SCS求解连续问题
problem_cont = cp.Problem(objective, constraints)
problem_cont.solve(solver=cp.SCS, verbose=True)

if problem_cont.status == cp.OPTIMAL:
    continuous_solution = x_cont.value
    print(f"连续问题最优值: {problem_cont.value:.2f}")
    print(f"连续问题最优解: {continuous_solution}")
    
    # 目标函数计算
    def objective_func(x):
        return (c1[0]*x[0]**2 + c1[1]*x[1]**2 + c1[2]*x[2]**2 + c1[3]*x[3]**2 + c1[4]*x[4]**2 +
                c2[0]*x[0] + c2[1]*x[1] + c2[2]*x[2] + c2[3]*x[3] + c2[4]*x[4])
    
    # 约束检查
    def check_constraints(x):
        return (all(0 <= xi <= 99 for xi in x) and
                A[0] @ x <= b[0] and
                A[1] @ x <= b[1] and
                A[2] @ x <= b[2] and
                A[3] @ x <= b[3])
    
    print("\n第二步：在连续解附近搜索整数解...")
    
    # 四舍五入得到初始整数解
    initial_int = np.round(continuous_solution).astype(int)
    print(f"四舍五入初始解: {initial_int}")
    
    # 搜索附近整数解
    best_obj = float('inf')
    best_sol = None
    search_range = 3  # 搜索范围
    
    for dx1 in range(-search_range, search_range+1):
        for dx2 in range(-search_range, search_range+1):
            for dx3 in range(-search_range, search_range+1):
                for dx4 in range(-search_range, search_range+1):
                    for dx5 in range(-search_range, search_range+1):
                        candidate = initial_int + np.array([dx1, dx2, dx3, dx4, dx5])
                        if check_constraints(candidate):
                            obj_val = objective_func(candidate)
                            if obj_val < best_obj:
                                best_obj = obj_val
                                best_sol = candidate.copy()
                                print(f"找到更好解: 值={best_obj:.2f}, 解={best_sol}")
    
    if best_sol is not None:
        print("\n最终整数解:")
        print(f"最优值: {best_obj}")
        print(f"最优解: {best_sol}")
        print(f"约束验证:")
        print(f"  x1+x2+x3+x4+x5 = {sum(best_sol)} <= 400")
        print(f"  x1+2x2+2x3+x4+6x5 = {best_sol[0] + 2*best_sol[1] + 2*best_sol[2] + best_sol[3] + 6*best_sol[4]} <= 800")
        print(f"  2x1+x2+6x3 = {2*best_sol[0] + best_sol[1] + 6*best_sol[2]} <= 200")
        print(f"  x3+x4+5x5 = {best_sol[2] + best_sol[3] + 5*best_sol[4]} <= 200")
    else:
        print("未找到满足约束的整数解")

else:
    print("连续问题求解失败")

(CVXPY) Aug 25 11:42:15 PM: Your problem has 5 variables, 14 constraints, and 0 parameters.
(CVXPY) Aug 25 11:42:15 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Aug 25 11:42:15 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Aug 25 11:42:15 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Aug 25 11:42:15 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Aug 25 11:42:15 PM: Compiling problem (target solver=SCS).
(CVXPY) Aug 25 11:42:15 PM: Reduction chain: Dcp2Cone -> CvxAttr2Constr -> ConeMatrixStuffing -> SCS
(CVXPY) Aug 25 11:42:15 PM: Applying reduction Dcp2Cone
(CVXPY) Aug 25 11:42:15 PM: Applying reduction CvxAttr2Constr
(CVXPY) Aug 25 11:42:15 PM: Applying reduction ConeMatrixStuffing
(CVXPY) Aug 25 11:42:15 PM: Applying reduction SCS
(CVXPY) Aug 25 11:42:15 PM: Finished problem compilation (took 1.931e

第一步：求解连续松弛问题...
                                     CVXPY                                     
                                     v1.7.1                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
------------------------------------------------------------------
	       SCS v3.2.8 - Splitting Conic Solver
	(c) Brendan O'Donoghue, Stanford University, 2012
------------------------------------------------------------------
problem:  variables n: 10, constraints m: 19
cones: 	  z: primal zero / dual free vars: 5
	  l: linear vars: 14
set

### 6.2.3 飞行管理问题  
在约 $ 10\ \text{km} $ 高空的某边长 $ 160\ \text{km} $ 的正方形区域内，经常有若干架飞机管理问题做水平飞行。区域内每架飞机管理问题的位置和速度向量均由计算机记录其数据，以便进行飞行管理。当一架欲进入该区域的飞机管理问题到达区域边缘时，记录其数据后，要立即计算并判断是否会与区域内的飞机管理问题发生碰撞。如果会发生碰撞，则应计算如何调整各架（包括新进入的）飞机管理问题飞行的方向角，以避免碰撞。现假定条件如下：  

(1) 不碰撞的标准为任意两架飞机管理问题的距离大于 $ 8\ \text{km} $；  
(2) 飞机管理问题飞行方向角调整的幅度不应超过 $ 30^\circ $；  
(3) 所有飞机管理问题飞行速度均为 $ 800\ \text{km/h} $；  
(4) 进入该区域的飞机管理问题在到达区域边缘时，与区域内飞机管理问题的距离应在 $ 60\ \text{km} $ 以上；  
(5) 最多需考虑 $ 6 $ 架飞机管理问题；  
(6) 不必考虑飞机管理问题离开此区域后的状况。  


请对这个避免碰撞的飞行管理问题建立数学模型，列出计算步骤，对以下数据进行计算（方向角误差不超过 $ 0.01^\circ $），要求飞机管理问题飞行方向角调整的幅度尽量小。  


设该区域 $ 4 $ 个顶点的坐标为 $ (0,0),(160,0),(160,160),(0,160) $。记录数据见表 6.3。  


#### 表 6.3 飞行记录数据  

| 飞机管理问题编号 | 横坐标 $ x $ | 纵坐标 $ y $ | 方向角$/(^\circ)$ |
| ---- | ---- | ---- | ---- |
| 1 | 150 | 140 | 243 |
| 2 | 85 | 85 | 236 |
| 3 | 150 | 155 | 220.5 |
| 4 | 145 | 50 | 159 |
| 5 | 130 | 150 | 230 |
| 新进入 | 0 | 0 | 52 |  


**注**：方向角指飞行方向与 $ x $ 轴正向的夹角。  


为方便以后的讨论，引进如下记号：  

- $ a $ 为飞机管理问题飞行速度，$ a = 800\ \text{km/h} $；  
- $ (x_i^0, y_i^0) $ 为第 $ i $ 架飞机管理问题的初始位置，$ i = 1,\cdots,6 $，$ i = 6 $ 对应新进入的飞机管理问题；  
- $ (x_i(t), y_i(t)) $ 为第 $ i $ 架飞机管理问题在 $ t $ 时刻的位置；  
- $ \theta_i^0 $ 为第 $ i $ 架飞机管理问题的原飞行方向角，即飞行方向与 $ x $ 轴夹角，$ 0 \leqslant \theta_i^0 < 2\pi $；  
- $ \Delta\theta_i $ 为第 $ i $ 架飞机管理问题的方向角调整量，$ -\frac{\pi}{6} \leqslant \Delta\theta_i \leqslant \frac{\pi}{6} $；  
- $ \theta_i = \theta_i^0 + \Delta\theta_i $ 为第 $ i $ 架飞机管理问题调整后的飞行方向角。  


#### 模型建立  
根据相对运动的观点在考察两架飞机管理问题 $ i $ 和 $ j $ 的飞行时，可以将飞机管理问题 $ i $ 视为不动而飞机管理问题 $ j $ 以相对速度  
$$
v = v_j - v_i = \big(a\cos\theta_j - a\cos\theta_i,\ a\sin\theta_j - a\sin\theta_i\big), \tag{6.6}
$$  

相对于飞机管理问题 $ i $ 运动，对 (6.6) 式进行适当的计算可得  
$$
\begin{align*}
v &= 2a\sin\frac{\theta_j - \theta_i}{2} \left( -\sin\frac{\theta_j + \theta_i}{2},\ \cos\frac{\theta_j + \theta_i}{2} \right) \\
&= 2a\sin\frac{\theta_j - \theta_i}{2} \left( \cos\left( \frac{\pi}{2} + \frac{\theta_j + \theta_i}{2} \right),\ \sin\left( \frac{\pi}{2} + \frac{\theta_j + \theta_i}{2} \right) \right), \tag{6.7}
\end{align*}
$$  

不妨设 $ \theta_j \geqslant \theta_i $，此时相对飞行方向角为 $ \beta_{ij} = \frac{\pi}{2} + \frac{\theta_i + \theta_j}{2} $，见图 6.1。  


由于两架飞机管理问题的初始距离为  
$$
r_{ij}(0) = \sqrt{(x_i^0 - x_j^0)^2 + (y_i^0 - y_j^0)^2}, \tag{6.8}
$$  

$$
\alpha_{ij}^0 = \arcsin\frac{8}{r_{ij}(0)}, \tag{6.9}
$$  

于是，只要当相对飞行方向角 $ \beta_{ij} $ 满足  
$$
\alpha_{ij}^0 < \beta_{ij} < 2\pi - \alpha_{ij}^0 \tag{6.10}
$$  

时，两架飞机管理问题不可能碰撞（图 6.1）。  


记 $ \beta_{ij}^0 $ 为调整前第 $ j $ 架飞机管理问题相对于第 $ i $ 架飞机管理问题的相对速度（矢量）与这两架飞机管理问题连线（从 $ j $ 指向 $ i $ 的矢量）的夹角（以连线矢量为基准，逆时针方向为正，顺时针方向为负）。则由式 (6.10) 知，两架飞机管理问题不碰撞的条件为  
$$
\left| \beta_{ij}^0 + \frac{1}{2}(\Delta\theta_i + \Delta\theta_j) \right| > \alpha_{ij}^0, \quad i = 1,\cdots,5,\ j = i + 1,\cdots,6, \tag{6.11}
$$  


其中  
$$
\begin{align*}
\beta_{mn}^0 &= \text{相对速度 } v_{mn} \text{ 的辐角 - 从 } n \text{ 指向 } m \text{ 的连线矢量的辐角} \\
&= \arg \frac{e^{i\theta_n^0} - e^{i\theta_m^0}}{(x_m + iy_m) - (x_n + iy_n)}.
\end{align*}
$$  


**注意**：$ \beta_{mn}^0 $ 表达式中的 $ i $ 表示虚数单位，这里为了区别虚数单位 $ i $ 或 $ \text{j} $，下标改写成 $ m,n $。这里利用复数的辐角，可以很方便地计算角度 $ \beta_{mn}^0\ (m,n = 1,2,\cdots,6) $。  


本问题中的优化目标函数可以有不同的形式：如使所有飞机管理问题的最大调整量最小；所有飞机管理问题的调整量绝对值之和最小等。这里以所有飞机管理问题的调整量绝对值之和最小为目标函数，可以得到如下的数学规划模型  
$$
\begin{align*}
\min &\quad \sum_{i=1}^{6} |\Delta\theta_i|, \\
\text{s.t.} &\quad 
\begin{cases} 
\left| \beta_{ij}^0 + \frac{1}{2}(\Delta\theta_i + \Delta\theta_j) \right| > \alpha_{ij}^0, & i = 1,\cdots,5,\ j = i + 1,\cdots,6, \\
|\Delta\theta_i| \leqslant 30^\circ, & i = 1,2,\cdots,6.
\end{cases}
\end{align*}
$$

In [ ]:
import numpy as np
import pandas as pd

x0 = np.array([150, 85, 150, 145, 130, 0])
y0 = np.array([140, 85, 155, 50, 150, 0])
q = np.array([243, 236, 220.5, 159, 230, 52])
d = np.zeros((6, 6))  # distance
a0 = np.zeros((6, 6))
b0 = np.zeros((6, 6))
xy0 = np.c_[x0, y0]  # 坐标(x, y)
for i in range(6):
    for j in range(6):
        d[i, j] = np.linalg.norm(xy0[i] - xy0[j])
d[np.where(d==0)] = np.inf  # 自己与自己距离无穷
a0 = np.arcsin(8 / d) * 180 / np.pi  # 转化为弧度
xy1 = x0 + 1j * y0  # 分母
xy2 = np.exp(1j * q * np.pi / 180)  # q*np.pi/180方向角转化为弧度，分子，欧拉公式
for m in range(6): 
    for n in range(6):
        if n != m:
            b0[m, n] = np.angle((xy2[n] - xy2[m]) / (xy1[m] - xy1[n]))
b0 = b0 * 180 / np.pi  # 角度
with pd.ExcelWriter("飞机管理问题管理问题.xlsx") as f:
    pd.DataFrame(a0).to_excel(f, sheet_name="alpha_ij", index=False)
    pd.DataFrame(b0).to_excel(f, sheet_name="Beta_mn", index=False)

利用上述 Python 程序，求得 $ \alpha_{ij}^0 $ 的值如表 6.4 所示。求得 $ \beta_{ij}^0 $ 的值如表 6.5 所示。  


### 表 6.4 $ \boldsymbol{\alpha_{ij}^0} $ 的值  

|    | 1 | 2 | 3 | 4 | 5 | 6 |
| ---- | ---- | ---- | ---- | ---- | ---- | ---- |
| 1 | 0 | 5.39119 | 32.23095 | 5.091816 | 20.96336 | 2.234507 |
| 2 | 5.39119 | 0 | 4.804024 | 6.61346 | 5.807866 | 3.815925 |
| 3 | 32.23095 | 4.804024 | 0 | 4.364672 | 22.83365 | 2.125539 |
| 4 | 5.091816 | 6.61346 | 4.364672 | 0 | 4.537692 | 2.989819 |
| 5 | 20.96336 | 5.807866 | 22.83365 | 4.537692 | 0 | 2.309841 |
| 6 | 2.234507 | 3.815925 | 2.125539 | 2.989819 | 2.309841 | 0 |  


### 表 6.5 $ \boldsymbol{\beta_{ij}^0} $ 的值  

|    | 1 | 2 | 3 | 4 | 5 | 6 |
| ---- | ---- | ---- | ---- | ---- | ---- | ---- |
| 1 | 0 | 109.26 | -128.25 | 24.18 | 173.07 | 14.475 |
| 2 | 109.26 | 0 | -88.871 | -42.244 | -92.305 | 9 |
| 3 | -128.25 | -88.871 | 0 | 12.476 | -58.786 | 0.31081 |
| 4 | 24.18 | -42.244 | 12.476 | 0 | 5.9692 | -3.5256 |
| 5 | 173.07 | -92.305 | -58.786 | 5.9692 | 0 | 1.9144 |
| 6 | 14.475 | 9 | 0.31081 | -3.5256 | 1.9144 | 0 |

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

a0 = pd.read_excel("飞机管理问题.xlsx")  # 读取表1,alpha_ij
b0 = pd.read_excel("飞机管理问题.xlsx", 1)  # 读取表2,beta_mn
a0 = a0.values  # 转化为ndarray
b0 = b0.values
obj = lambda x: np.sum(np.abs(x))  # 目标函数
bd = [(-30, 30) for i in range(6)]  # 上下界
x0 = np.ones((1, 6))  # 决策变量
cons = []
for i in range(5):
    for j in range(i+1, 6):  # i!=j
        cons.append({'type': 'ineq', 'fun': lambda x: np.abs(b0[i, j] + (x[i] + x[j]) / 2) - a0[i, j]})
res = minimize(obj, np.ones(6), constraints=cons, bounds=bd)
print(res)

 message: Optimization terminated successfully
 success: True
  status: 0
     fun: 0.7909172985328354
       x: [ 2.533e-07  2.533e-07  2.533e-07  2.533e-07  3.955e-01
            3.955e-01]
     nit: 6
     jac: [-1.000e+00 -1.000e+00 -1.000e+00 -1.000e+00  1.000e+00
            1.000e+00]
    nfev: 54
    njev: 6


利用上述 Python 程序，求得的最优解为 $ \Delta\theta_3 = 0.3955^\circ $，$ \Delta\theta_6 = 0.3955^\circ $，其他调整角度为 0，总的调整角度为 $ 0.7909^\circ $。